# S7.G1 · B_T1 叶子编码组合披露评估

> 步骤声明：`modules/m7_security/S7.G1_B_T1披露评估_步骤声明.md` ｜ 配置：`modules/m7_security/configs/s7_g1_bt1_disclosure.yaml` ｜ 输入：S5.G1 私有数组（`G3_PRIVATE_ROOT`）

B_T1 由香港对每位对齐客户发出 k 粗化后的随机树叶子编码。内地本来就知道每行是哪位客户，所以这里度量的是：**编码让内地对一位已识别客户的香港字段知道了多少**。推断按最坏情形计——假设内地知道编码到特征区域的映射（知道树结构，或有少量已知香港字段的辅助客户）。

开发评估集只用其香港特征度量发布内容，不用标签；披露—效用权衡只在内层验证集上计算。公开竞赛数据上的构造切分，不代表汇丰真实效果；技术侧自评，未经法务复核（W-001）。

In [1]:
import hashlib
import json
import os
import pathlib
import subprocess
import sys

import numpy as np
import pandas as pd
import yaml
from sklearn.metrics import roc_auc_score

HASH_PREFIX_LEN = 12

REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

CONFIG_PATH = REPO_ROOT / "modules/m7_security/configs/s7_g1_bt1_disclosure.yaml"
cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
run = yaml.safe_load((REPO_ROOT / cfg["source_run_config"]).read_text(encoding="utf-8"))
PRE_PATH = REPO_ROOT / cfg["preregistration"]
pre = yaml.safe_load(PRE_PATH.read_text(encoding="utf-8"))
fz = pre["freeze"]
ad = fz["arm_details"]


def sha(path, n=HASH_PREFIX_LEN):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()[:n]


git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True, cwd=REPO_ROOT).strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                       cwd=REPO_ROOT).stdout.strip() != ""
SEEDS = pre["seeds"]["training"]
print("config:", CONFIG_PATH.relative_to(REPO_ROOT), "sha256:" + sha(CONFIG_PATH))
print("preregistration:", PRE_PATH.relative_to(REPO_ROOT), "sha256:" + sha(PRE_PATH), "| status:", pre["status"])
print("seeds:", SEEDS)
print("git:", git_sha + ("（工作区含本步未提交改动）" if dirty else ""))
print("step:", cfg["step_id"], "|", cfg["purpose"])
mismatch = [p for p, h in fz["code_sha256"].items() if hashlib.sha256((REPO_ROOT / p).read_bytes()).hexdigest() != h]
print("冻结代码指纹核对：", "一致" if not mismatch else "不一致 " + str(mismatch))
PRIV = pathlib.Path(os.environ[cfg["private_root_env"]])
bad = [f for f, h in cfg["private_inputs_sha256"].items() if sha(PRIV / f, None) != h]
print("私有输入指纹核对（S5.G1 产物）：", "一致" if not bad else "不一致 " + str(bad))
assert pre["status"] == "frozen" and not mismatch and not bad
MB, NJ, K = ad["xgboost_max_bin"], ad["n_jobs"], ad["b_t1"]["k"]


config: modules/m7_security/configs/s7_g1_bt1_disclosure.yaml sha256:2aed5e958a2b
preregistration: modules/m6_evaluation/configs/s6_p3_g3_preregistration.yaml sha256:c0a4bbfc9709 | status: frozen
seeds: [11, 22, 33, 44, 55, 66]
git: a48b16d（工作区含本步未提交改动）
step: S7.G1 | 度量 B_T1 发出的叶子编码在编码组合层面的等价类规模、内地据此推断香港字段的能力，并在内层验证集上给出披露与效用的权衡
冻结代码指纹核对： 一致
私有输入指纹核对（S5.G1 产物）： 一致


**读法**：首格核对预注册仍为 frozen、冻结组件与 S5.G1 私有输入的指纹一致。notebook 在提交前的工作区执行，指纹打印父提交加「未提交改动」标记。

In [2]:
from modules.m5_modeling.components import g3_arms as A
from modules.m5_modeling.components import g3_santander as G
from modules.m7_security.components import code_disclosure as D

arr = np.load(PRIV / run["arrays_file"])
SPLITS = cfg["release_splits"]
X = {n: {k: arr[n + "__" + k] for k in ("xg", "xh", "y", "cust")} for n in SPLITS}
tuning = json.loads((PRIV / run["tuning_file"]).read_text(encoding="utf-8"))
saved = np.load(PRIV / run["local_scores_file"])
XH = {n: X[n]["xh"] for n in SPLITS}
CT = X["train"]["cust"]


def bt1_features(coarse):
    return {n: np.hstack([X[n]["xg"], D.one_hot_release(coarse[n], coarse["train"])]) for n in coarse}


# 复现核对 ①：B_T1 第一个 seed 的开发评估集打分（只用特征）
s0 = SEEDS[cfg["reproduce_seed_index"]]
b = ad["b_t1"]
REL = D.release_codes(XH, CT, b["n_trees"], b["depth"], K, s0, tuple_k=False)
F = bt1_features(REL)
m = A.xgb_fit(F["train"], X["train"]["y"], tuning["B_T1"]["best"], MB, s0, NJ)
diff_bt1 = float(np.max(np.abs(A.xgb_score(m, F["dev_eval"]) - saved["B_T1"][cfg["reproduce_seed_index"]])))

# 复现核对 ②：L1_k10 的分段均值发布（对照组），用 S5.G1 的做法重建后比对开发评估集打分
cols = {name: G.GUEST_FEATURES.index(col) for name, col in run["segment_columns"].items()}
cols.update({c: G.GUEST_FEATURES.index(c) for c in run["segment_categorical"]})
edges = A.segment_edges(X["train"]["xg"], cols, ad["l1_segments"]["quantile_bins"])
cat_sizes = {c: int(X["train"]["xg"][:, cols[c]].max()) + 1 for c in run["segment_categorical"]}
seg = {n: A.segment_ids(X[n]["xg"], cols, edges, cat_sizes) for n in SPLITS}
table, gmean, _ = A.k_anonymous_means(seg["train"], CT, X["train"]["xh"], cfg["l1_comparison_k"])
SEGM = {n: A.apply_segment_means(seg[n], table, gmean) for n in SPLITS}
l1_arm = "L1_k%d" % cfg["l1_comparison_k"]
m = A.xgb_fit(np.hstack([X["train"]["xg"], SEGM["train"]]), X["train"]["y"], tuning[l1_arm]["best"], MB, s0, NJ)
diff_l1 = float(np.max(np.abs(A.xgb_score(m, np.hstack([X["dev_eval"]["xg"], SEGM["dev_eval"]])) - saved[l1_arm])))
print("复现最大绝对差：B_T1 %.2e，%s %.2e（容差 %.0e）" % (diff_bt1, l1_arm, diff_l1, cfg["reproduce_tolerance"]))
assert max(diff_bt1, diff_l1) <= cfg["reproduce_tolerance"]


复现最大绝对差：B_T1 0.00e+00，L1_k10 0.00e+00（容差 1e-06）


**读法**：按 S5.G1 的做法重建的 B_T1（第一个 seed）与 L1_k10 发布内容，在开发评估集上的打分与 S5.G1 保存的打分逐行相同（最大绝对差 0）。下文度量的就是 S5.G1 实际用到的发布内容。

In [3]:
# S5.G1 实际发布的 B_T1（10 棵深 3、单棵树 k = 10），6 个 seed
uniq_rows, inf = [], {"auc": [], "certain_share": [], "positive_determined": [], "certain_accuracy": []}
unseen, near = [], []
for s in SEEDS:
    rel = D.release_codes(XH, CT, b["n_trees"], b["depth"], K, s, tuple_k=False)
    for n in SPLITS:
        uniq_rows.append(dict(D.uniqueness(rel[n], X[n]["cust"], K), split=n, seed=s))
    out = D.disclosure_summary(rel[cfg["mapping_split"]], XH[cfg["mapping_split"]], rel["dev_eval"], XH["dev_eval"])
    for key in inf:
        inf[key].append(out[key])
    unseen.append(float(out["unseen_tuple_share"][0]))
    near.append(int(np.sum(out["auc"] >= cfg["near_certain_auc"])))
U = pd.DataFrame(uniq_rows)
share_cols = ["row_share_class_lt_k", "row_share_unique", "customer_share_any_row_lt_k"]
uniq = U.groupby("split", sort=False).agg(
    distinct_tuples=("distinct_tuples", "mean"), median_class_size=("median_class_size", "mean"),
    **{c: (c, "mean") for c in share_cols}, **{c + "_sd": (c, "std") for c in share_cols}).reset_index()
print(uniq.round(cfg["round_share"]).to_string(index=False))

attr = pd.DataFrame({"hk_field": G.HOST_FEATURES})
for key, vals in inf.items():
    attr["bt1_" + key], attr["bt1_" + key + "_sd"] = D.mean_over(vals)
attr["l1_auc"] = D.segment_disclosure(SEGM["dev_eval"], XH["dev_eval"])
attr["positive_rate_dev_eval"] = [float((v[v != D.MISSING] == 1.0).mean()) for v in XH["dev_eval"].T]
NC_AUC = cfg["near_certain_auc"]
print(attr[["hk_field", "positive_rate_dev_eval", "bt1_auc", "l1_auc", "bt1_positive_determined",
            "bt1_certain_share"]].round(cfg["round_auc"]).to_string(index=False))
print("字段数 %d。单次发布（每个 seed 一次）推断 AUROC ≥ %.2f 的字段：%s，均值 %.1f；L1 %d 个"
      % (len(attr), NC_AUC, near, float(np.mean(near)), int((attr["l1_auc"] >= NC_AUC).sum())))
print("6 个 seed 的均值 AUROC ≥ %.2f 的字段 %d 个（各 seed 的随机树选中的字段不同）；字段平均 AUROC：B_T1 %.4f，L1 %.4f"
      % (NC_AUC, int((attr["bt1_auc"] >= NC_AUC).sum()), attr["bt1_auc"].mean(), attr["l1_auc"].mean()))
print("持有者被确定推出的比例（字段平均）：B_T1 %.4f；开发评估集组合未在训练发布中出现的行：%.4f"
      % (attr["bt1_positive_determined"].mean(), float(np.mean(unseen))))


    split  distinct_tuples  median_class_size  row_share_class_lt_k  row_share_unique  customer_share_any_row_lt_k  row_share_class_lt_k_sd  row_share_unique_sd  customer_share_any_row_lt_k_sd
    train         557.6667          2172.0000                0.0535            0.0151                       0.0655                   0.0095               0.0043                          0.0098
inner_val         428.6667          2911.5000                0.0532            0.0149                       0.0613                   0.0101               0.0046                          0.0100
 dev_eval         501.0000          5990.8333                0.0439            0.0123                       0.0439                   0.0099               0.0034                          0.0099


             hk_field  positive_rate_dev_eval  bt1_auc  l1_auc  bt1_positive_determined  bt1_certain_share
    ind_ahor_fin_ult1                  0.0000   0.5000  0.5000                   0.0000             0.9878
    ind_aval_fin_ult1                  0.0000   0.5000  0.5000                   0.0000             0.9878
     ind_cco_fin_ult1                  0.6112   0.9601  0.6549                   0.8215             0.8226
    ind_cder_fin_ult1                  0.0003   0.8895  0.5056                   0.3333             0.9167
     ind_cno_fin_ult1                  0.0786   0.9980  0.7045                   0.9018             0.9357
    ind_ctju_fin_ult1                  0.0074   0.9957  0.9979                   0.8333             0.9446
    ind_ctma_fin_ult1                  0.0086   0.9598  0.7104                   0.7812             0.8518
    ind_ctop_fin_ult1                  0.1115   0.9477  0.7827                   0.7778             0.8097
    ind_ctpp_fin_ult1                

**读法**（6 个 seed 的均值；推断映射从训练发布中学，在开发评估集客户上度量）：

- **编码组合层面不满足 k-匿名（R1 成立）**：训练发布中约 5.4% 的行、6.5% 的客户所在组合不足 10 位不同客户，约 1.5% 的行组合唯一；开发评估集为 4.4%。
- **单次发布平均有 16.2 个字段（6 个 seed 为 14–20 个）可被几乎确定地推出（AUROC ≥ 0.99），超过 25 个字段的一半（R2 成立）。**各 seed 的随机树选中的字段不同，所以 6 个 seed 均值 ≥ 0.99 的只有 5 个：`ind_actividad_cliente`（活跃度）、`ind_nomina_ult1`（工资代发）、`ind_cno_fin_ult1`（工资账户）、`ind_nom_pens_ult1`（养老金代发）、`ind_ctju_fin_ult1`（青少年账户）。
- 持有者被确定推出的比例，字段平均 59%；活期账户 `ind_cco_fin_ult1` 82%、工资代发 90%、养老金代发 88%、活跃度 96%。
- 对照：L1_k10 的分段均值作推断时，字段平均 AUROC 0.716，只有青少年账户 ≥ 0.99——它由年龄决定，而年龄本来就是内地的特征。
- 这些推断假设内地知道编码到特征区域的映射（知道树结构，或有少量已知香港字段的辅助客户）；没有辅助信息时编码本身不透明。

In [4]:
# 披露—效用权衡：只在内层验证集上；XGBoost 配置固定为 S5.G1 选定的 B_T1 配置
TR = {n: XH[n] for n in ("train", "inner_val")}
yv = X["inner_val"]["y"]
rows = []
for v in cfg["tradeoff_variants"]:
    for s in SEEDS:
        rel = D.release_codes(TR, CT, v["n_trees"], v["depth"], K, s, tuple_k=v["tuple_k"])
        F = bt1_features(rel)
        m = A.xgb_fit(F["train"], X["train"]["y"], tuning["B_T1"]["best"], MB, s, NJ)
        u = D.uniqueness(rel["train"], CT, K)
        out = D.disclosure_summary(rel["train"], TR["train"], rel["inner_val"], TR["inner_val"])
        rows.append({"variant": v["name"], "seed": s, "inner_val_auc": roc_auc_score(yv, A.xgb_score(m, F["inner_val"])),
                     "released_columns": F["train"].shape[1] - X["train"]["xg"].shape[1],
                     "row_share_class_lt_k": u["row_share_class_lt_k"],
                     "field_auc_mean": float(np.nanmean(out["auc"])),
                     "fields_near_certain": int(np.sum(out["auc"] >= NC_AUC)),
                     "positive_determined_mean": float(np.nanmean(out["positive_determined"]))})
T = pd.DataFrame(rows)
metric_cols = ["inner_val_auc", "released_columns", "row_share_class_lt_k", "field_auc_mean",
               "fields_near_certain", "positive_determined_mean"]
trade = T.groupby("variant", sort=False)[metric_cols].mean()
trade["inner_val_auc_sd"] = T.groupby("variant", sort=False)["inner_val_auc"].std()
l1_dis = D.segment_disclosure(SEGM["inner_val"], XH["inner_val"])
refs = pd.DataFrame([{"variant": a, "inner_val_auc": tuning[a]["best_val_auc"]} for a in ("L0", l1_arm)]).set_index("variant")
refs.loc[l1_arm, "field_auc_mean"] = float(np.nanmean(l1_dis))
refs.loc[l1_arm, "fields_near_certain"] = int(np.sum(l1_dis >= NC_AUC))
trade = pd.concat([trade, refs]).reset_index().rename(columns={"index": "variant"})
print(trade.round(cfg["round_auc"]).to_string(index=False))


  variant  inner_val_auc  released_columns  row_share_class_lt_k  field_auc_mean  fields_near_certain  positive_determined_mean  inner_val_auc_sd
    T1_d2         0.8208            3.5000                0.0001          0.6446               1.8333                    0.0892            0.0116
    T3_d2         0.8365           10.8333                0.0018          0.7313               4.8333                    0.2382            0.0211
    T3_d3         0.8503           18.6667                0.0075          0.8003               6.5000                    0.3415            0.0141
   T10_d2         0.8669           35.5000                0.0408          0.8901              13.8333                    0.5583            0.0117
   T10_d3         0.8782           60.0000                0.0535          0.9340              15.8333                    0.6266            0.0037
 T3_d3_tk         0.8500           17.0000                0.0000          0.8064               6.8333                    0.2

**读法**（内层验证集；它也用于 S5.G1 的调参与 B_best 选择，数值偏乐观，只作变体间的相对比较）：

- **精度与披露同升同降**：从 1 棵深 2 到 10 棵深 3，内层验证 AUROC 0.821 → 0.878，几乎确定可推出的字段 1.8 → 15.8 个，持有者确定推出比例 9% → 63%。
- **组合级 k-匿名抑制**把「组合不足 k」的行降到 0，精度代价很小（10 棵深 3：0.878 → 0.874；3 棵深 3：0.850 → 0.850），持有者确定推出比例 63% → 42%（10 棵）、34% → 29%（3 棵）；但几乎确定可推出的字段只从 15.8 降到 11.3（10 棵），3 棵时不降（6.5 → 6.8）。组合级 k-匿名解决的是「组合指向个人」，对已识别客户的字段推断作用有限。
- **R3**：3 棵深 3（含其组合级抑制版）内层验证 AUROC 0.850，比 L1_k10（0.818）高约 0.03，几乎确定可推出的字段 6.5–6.8 个，少于已发布 B_T1 的一半（15.8）——记为值得在开发评估集上预注册比较的候选，本步不评估它。它仍逐客户发出约 7 个可确定推出的香港字段。

In [5]:
out_dir = REPO_ROOT / cfg["results_dir"]
out_dir.mkdir(parents=True, exist_ok=True)
uniq.assign(fields_near_certain_per_release_mean=float(np.mean(near)), fields_near_certain_min=min(near),
            fields_near_certain_max=max(near)).round(cfg["round_share"]).to_csv(out_dir / "released_uniqueness.csv", index=False)
attr.round(cfg["round_auc"]).to_csv(out_dir / "attribute_inference_dev_eval.csv", index=False)
trade.round(cfg["round_auc"]).to_csv(out_dir / "tradeoff_inner_val.csv", index=False)
print("已写出汇总表：", sorted(p.name for p in out_dir.glob("*.csv")))


已写出汇总表： ['attribute_inference_dev_eval.csv', 'released_uniqueness.csv', 'tradeoff_inner_val.csv']


## 结论

1. **B_T1 不是匿名统计。** 编码组合层面不满足 k-匿名；更关键的是内地本来就知道每行是谁——单次发布平均有 16 个香港字段可被几乎确定地推出。B_T1 实质上是把多数香港产品持有字段（粗化后）逐客户发给内地。
2. **S5.G1 中「单向形态接近双向」的原因就在这里**：精度与披露同升同降。这条精度结论不能当作「单向形态披露更少」的证据（R2）。
3. **对 S0.9 形态判决的输入**：若采用单向形态，香港→内地发出的是关于已识别客户的个人资料，须按香港《个人资料（私隐）条例》评估香港行能否向内地行提供；内地→香港方向仍只有对号环节。双向 A 的推理期披露此前已测（INNOVATIONS A2，修复用 SPU 安全推理），本步没有在同一口径下度量 A，不比较两者的披露量。
4. **折中候选**：3 棵深 3 加组合级 k-匿名（内层验证 0.850，比 L1 高约 0.03，可确定推出的字段约 7 个）。若要采用，须另行预注册开发评估集上的比较。
5. **边界**：公开竞赛数据的构造切分；最坏情形假设内地知道映射；不代表汇丰真实效果；技术侧自评，未经法务复核（W-001）。